# Phase 6: Random Forest Baseline Model

**Project:** ML-Powered Intrusion Detection System (IDS) for Secure Network Monitoring  
**Phase:** Phase 6 — Random Forest Baseline Model  
**Dataset:** CICIoT2023 (Standardized Tabular Flow Features, 46 Features, 34 Classes)  

This notebook trains, evaluates, and diagnoses the Scikit-Learn **Random Forest** baseline model on the verified processed dataset partitions (`data/processed/`).

## 1. Setup & Environment Verification

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is accessible
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.random_forest import RandomForestBaselineModel
from src.preprocessing.verify_split import load_partition

print("Environment initialized successfully.")

## 2. Load Processed Datasets & Verify Partitions

In [ ]:
data_dir = PROJECT_ROOT / "data" / "processed"

X_train, y_train = load_partition(data_dir / "train", "train")
X_val, y_val = load_partition(data_dir / "validation", "val")
X_test, y_test = load_partition(data_dir / "test", "test")

with open(data_dir / "label_mapping.json", "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

inv_mapping = {v: k for k, v in label_mapping.items()}
target_names = [inv_mapping[i] for i in range(len(label_mapping))]

with open(data_dir / "preprocessing_metadata.json", "r", encoding="utf-8") as f:
    meta = json.load(f)
    feature_names = meta.get("feature_names", [f"feature_{i}" for i in range(X_train.shape[1])])

print(f"Train Partition:      {X_train.shape[0]:,} samples | {X_train.shape[1]} features")
print(f"Validation Partition: {X_val.shape[0]:,} samples | {X_val.shape[1]} features")
print(f"Test Partition:       {X_test.shape[0]:,} samples | {X_test.shape[1]} features")
print(f"Distinct Classes:     {len(target_names)}")

## 3. Display Class Distribution & Class Weighting Rationale

In [ ]:
unique_train, counts_train = np.unique(y_train, return_counts=True)
dist_df = pd.DataFrame({
    "Class_Index": unique_train,
    "Class_Name": [inv_mapping[c] for c in unique_train],
    "Train_Count": counts_train,
    "Percentage": (counts_train / len(y_train)) * 100
}).sort_values(by="Train_Count", ascending=False)

display(dist_df.head(10))
display(dist_df.tail(10))

print(f"Imbalance Ratio (Max/Min): {dist_df['Train_Count'].max() / dist_df['Train_Count'].min():.1f}:1")
print("Justification: Class weights set to 'balanced' to penalize misclassifications on rare attack classes.")

## 4. Instantiate & Train Random Forest Baseline Model

In [ ]:
rf_model = RandomForestBaselineModel(
    n_estimators=200,
    max_depth=25,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    max_samples=0.1,
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest classifier on training partition...")
t0 = time.time()
rf_model.fit(X_train, y_train)
train_time = time.time() - t0
print(f"Training finished in {train_time:.2f} seconds.")

## 5. Diagnostic Evaluation on Validation Set

In [ ]:
from scripts.train_random_forest import calculate_metrics_and_report

t0 = time.time()
y_val_pred = rf_model.predict(X_val)
val_time = time.time() - t0

val_metrics, val_report, _ = calculate_metrics_and_report(y_val, y_val_pred, target_names)
print(f"Validation Accuracy:          {val_metrics['accuracy']*100:.2f}%")
print(f"Validation Weighted F1-Score: {val_metrics['weighted_f1']:.4f}")
print(f"Validation Macro F1-Score:    {val_metrics['macro_f1']:.4f}")
print(f"Validation Inference Time:    {val_time:.4f}s")

## 6. Single Final Evaluation on Test Set

In [ ]:
t0 = time.time()
y_test_pred = rf_model.predict(X_test)
test_time = time.time() - t0

test_metrics, test_report, cm_test = calculate_metrics_and_report(y_test, y_test_pred, target_names)

print(f"Test Accuracy:          {test_metrics['accuracy']*100:.2f}%")
print(f"Test Weighted Precision:{test_metrics['weighted_precision']:.4f}")
print(f"Test Weighted Recall:   {test_metrics['weighted_recall']:.4f}")
print(f"Test Weighted F1-Score: {test_metrics['weighted_f1']:.4f}")
print(f"Test Macro Precision:   {test_metrics['macro_precision']:.4f}")
print(f"Test Macro Recall:      {test_metrics['macro_recall']:.4f}")
print(f"Test Macro F1-Score:    {test_metrics['macro_f1']:.4f}")
print(f"Test Inference Latency: {test_time:.4f}s")

## 7. Per-Class Classification Report

In [ ]:
display(test_report.sort_values(by="F1_Score", ascending=False))

## 8. Test Set Confusion Matrix

In [ ]:
plt.figure(figsize=(16, 14))
cm_norm = cm_test.astype('float') / (cm_test.sum(axis=1)[:, np.newaxis] + 1e-9)
sns.heatmap(
    cm_norm,
    annot=False,
    cmap="Blues",
    xticklabels=target_names,
    yticklabels=target_names,
    cbar_kws={'label': 'Normalized Prediction Ratio'}
)
plt.title("Random Forest Baseline — Normalized Test Confusion Matrix", fontsize=15)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("Ground Truth Label", fontsize=12)
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

## 9. Feature Importance Analysis

In [ ]:
feat_imp = rf_model.get_feature_importances(feature_names=feature_names)
feat_df = pd.DataFrame(list(feat_imp.items()), columns=["Feature", "Importance"]).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(12, 8))
top25 = feat_df.head(25).iloc[::-1]
plt.barh(top25["Feature"], top25["Importance"], color="#1f77b4")
plt.title("Top 25 Feature Importances (Gini Impurity)", fontsize=14)
plt.xlabel("Gini Importance", fontsize=12)
plt.ylabel("Feature", fontsize=12)
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 10. Summary & Artifact References

In [ ]:
print("Model saved to:       models/random_forest.pkl")
print("Metadata saved to:    models/random_forest_metadata.json")
print("Metrics saved to:     results/metrics/random_forest_test.json")
print("Comparison table:     results/metrics/model_comparison.csv")